In [1]:
import os
import sys

from libs.multilevel_squeme import partition_graph_metis

from libs.utils import (
    load_mtx,
    matrix_to_graph,
    summarize_generic
)


In [2]:
print("Python executable:", sys.executable)

try:
    import pymetis
    print("PyMetis:", getattr(pymetis, "__file__", "?"))
except Exception as e:
    print("PyMetis import failed:", repr(e))

Python executable: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/.venv/bin/python
PyMetis: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/.venv/lib/python3.10/site-packages/pymetis/__init__.py


In [3]:
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

k_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_k.mtx")
m_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_m.mtx")
print("Using data_dir:", data_dir)

Using data_dir: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/data


In [4]:
# Load matrices
A_K = load_mtx(k_matrix_path, 'K')
A_M = load_mtx(m_matrix_path, 'M')

if A_K is None or A_M is None:
    raise RuntimeError("Matrix load failed; check file paths printed above.")

print(f"Loaded: K | shape={A_K.shape}, nnz={A_K.nnz}")
print(f"Loaded: M | shape={A_M.shape}, nnz={A_M.nnz}")

Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098
Loaded: K | shape=(960, 960), nnz=30282
Loaded: M | shape=(960, 960), nnz=10098


In [5]:
# Build graphs from matrices
diag_K = A_K.diagonal()
diag_M = A_M.diagonal()

G_K = matrix_to_graph(A_K, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_K)
G_M = matrix_to_graph(A_M, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_M)

print(f"K: |V|={G_K.number_of_nodes()}, |E|={G_K.number_of_edges()}")
print(f"M: |V|={G_M.number_of_nodes()}, |E|={G_M.number_of_edges()} (vweights set from diagonal)")

K: |V|=960, |E|=14661
M: |V|=960, |E|=4569 (vweights set from diagonal)


In [6]:
# Partitioning configuration
# Modes: 'kway_direct' | 'kway_metis'
PARTITION_MODE = 'kway_direct'
NPARTS = 16  # K-way only

In [7]:
# Verbosity toggle for library pipelines
VERBOSE = False  # set to False to silence progress logs

In [8]:
# # Run partitions based on PARTITION_MODE and NPARTS
# # Default params per graph (lean core)
# params_K = dict(weight='weight', initial_method='GGGP', balance_tol=0.03, max_levels=12, coarsen_limit=8, refine_passes=25, n_trials=10, seed=42)
# params_M = dict(weight='weight', initial_method='GGGP', balance_tol=0.06, max_levels=10, coarsen_limit=7, refine_passes=30, n_trials=15, seed=123)

# if PARTITION_MODE == 'kway_direct':
#     # Direct K-way multilevel with K-way refinement and final rebalance
#     from libs.multilevel_squeme import multilevel_kway_partition
#     dparams_K = dict(params_K)
#     dparams_M = dict(params_M)
#     dparams_K['verbose'] = VERBOSE
#     dparams_M['verbose'] = VERBOSE
#     part_K = multilevel_kway_partition(G_K, NPARTS, **dparams_K)
#     part_M = multilevel_kway_partition(G_M, NPARTS, **dparams_M)
# elif PARTITION_MODE == 'kway_metis':
#     part_K = partition_graph_metis(G_K, nparts=NPARTS, weight='weight', seed=params_K.get('seed', 42), verbose=VERBOSE)
#     part_M = partition_graph_metis(G_M, nparts=NPARTS, weight='weight', seed=params_M.get('seed', 123), verbose=VERBOSE)
# else:
#     raise ValueError(f"Unsupported configuration: PARTITION_MODE={PARTITION_MODE}, NPARTS={NPARTS}")

# summarize_generic(G_K, part_K, f"K [{PARTITION_MODE} {NPARTS}]")
# summarize_generic(G_M, part_M, f"M [{PARTITION_MODE} {NPARTS}]")

In [9]:
# Baseline METIS comparison for the same NPARTS
metis_part_K = partition_graph_metis(G_K, nparts=NPARTS, weight='weight', seed=42, verbose=VERBOSE)
summarize_generic(G_K, metis_part_K, f"METIS K [{NPARTS}]")

metis_part_M = partition_graph_metis(G_M, nparts=NPARTS, weight='weight', seed=123, verbose=VERBOSE)
summarize_generic(G_M, metis_part_M, f"METIS M [{NPARTS}]")

METIS K [16]: cut=18127.3360, parts=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], per=[2101.324209690338, 2052.146135341699, 2063.166288877349, 2066.5010701710703, 2130.5845741601897, 2083.0383005070744, 2095.1148593938683, 2155.3410775614243, 2085.2365617989162, 2159.6697992685804, 2069.491596803579, 2110.1608488051465, 2164.1929841559913, 2093.0098363468087, 2054.9147893267277, 2161.9720416330892], total=33645.8650
Per difference: Max per: Min per: 112.04684881429239 2164.1929841559913 2052.146135341699
METIS M [16]: cut=31683.3174, parts=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], per=[11556.714731826425, 11319.049850993068, 11089.267586762524, 11571.32155683526, 11223.183117241142, 11045.891383541453, 11666.030247753944, 11253.13378740398, 11247.302117621102, 11628.19627658917, 11051.610355721534, 11060.421406732148, 11553.56009507756, 11039.559070658064, 11299.711329633068, 11617.929692673017], total=181222.8826
Per difference: Max per: Min per: 626.471177095